In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sqlite3
from estnltk import Text
from estnltk.taggers import VabamorfAnalyzer
import csv

### I Setup

In [12]:
RESULT_DATA_PATH = "C:/Users/liivas/Downloads/Töö/estnltk_projekt_2025/morphology_conflicts/data/"
TR_SOURCE_DATA_PATH = "C:/Users/liivas/Downloads/Töö/estnltk_projekt_2025/transaktsioonid/source_data/"
TR_DB = "v33_koondkorpus_sentences_verb_pattern_obl_20241002-130310.db"
SENTENCES_DB = "v33_koondkorpus_sentences_sentences_20250220-130121.db"
TR_RESULT_STRICT = "verb_obj_cases_strict.db"
#TR_RESULT_LENIENT = "verb_obj_cases_all.db"
EXAMPLE_RES = "verb_obj_case_examples_10.db"

### II Helper methods

In [4]:
morph_analyzer = VabamorfAnalyzer()

In [5]:
# detecting form homonymy
# NB! We're looking for specific cases

def has_form_homonymy(word_form: str) -> bool:
    nom_gen_part_adt = []
    analysis = morph_analyzer.analyze_token(word_form)
    for a_idx, a in enumerate(analysis):
        if a["form"] in ["sg n", "sg g", "sg p", "adt"]:
            nom_gen_part_adt.append(a["form"])
    if len(nom_gen_part_adt) > 1:
        return True
    else:
        return False

In [6]:
# getting POS of word form

def get_POS(word_form: str) -> str:
    pos = set()
    analysis = morph_analyzer.analyze_token(word_form)
    for a_idx, a in enumerate(analysis):
        pos.add(a["partofspeech"])
    pos_sorted = sorted(list(pos))
    return "|".join(pos_sorted)

In [7]:
# getting current case from feats field

def get_case(feats) -> str:
    cases = ["nom", 
             "gen", 
             "part", 
             "adit", 
             "ill", 
             "el", 
             "all", 
             "term", 
             "abl",
             "kom",
             "ad",
             "es",
             "abes",
             "tr"]
    
    feats_split = feats.split(",")
    case = ""
    # there should be 1 case per feats, but the order is not fixed
    for idx, feat in enumerate(feats_split):
        if feat in cases:
            case = feat
            break
    return case

In [8]:
# getting verb tense from feats field
# pres -> olevik
# past -> minevik

def get_verb_tense(feats: str) -> str:
    if "pres" in feats:
        return "pres"
    elif "past" in feats:
        return "past"
    else:
        return ""

In [9]:
# verb mood
# käskiv -> imper
# tingiv -> cond
# kindel -> indic
# kaudne -> quot
# möönev -> juss

# muud on partic (mittefiniitsed verbivormid)

def get_verb_mood(feats: str) -> str:
    if "imper" in feats:
        return "imper"
    elif "cond" in feats:
        return "cond"
    elif "indic" in feats:
        return "indic"
    elif "quot" in feats:
        return "quot"
    elif "juss" in feats:
        return "juss"
    else:
        if "partic" in feats:
            return "partic"
        else:
            return ""

### III Verbs, obj cases, frequencies in transactions DB (strict)

In [12]:
con = sqlite3.connect(f"{TR_SOURCE_DATA_PATH}{TR_DB}")

con.create_function("has_form_homonymy", 1, has_form_homonymy)
con.create_function("get_verb_tense", 1, get_verb_tense)
con.create_function("get_case", 1, get_case)
con.create_function("get_verb_mood", 1, get_verb_mood)

cur = con.cursor()
cur.execute(f'ATTACH DATABASE "{RESULT_DATA_PATH}{TR_RESULT_STRICT}" AS result')

cur.execute("""
DROP TABLE IF EXISTS result.verbs_obj_cases_lemma_freq
""")

cur.execute(
    """
    CREATE TABLE result.verbs_obj_cases_lemma_freq
    AS
    SELECT
        tr_head.verb AS verb,
        tr_head.verb_compound AS verb_compound,
        get_verb_tense(tr_head.feats) AS verb_tense,
        get_verb_mood(tr_head.feats) AS verb_mood,
        get_case(tr_row.feats) AS obj_case,
        count(DISTINCT tr_row.lemma) AS freq
    FROM
    (
        SELECT
            head_id,
            feats,
            lemma
        FROM
            transaction_row
        WHERE
            deprel = "obj"
        AND
            get_case(feats) IN ('nom', 'gen', 'part')
        AND NOT has_form_homonymy(form)
    ) AS tr_row
    INNER JOIN
        transaction_head AS tr_head
    ON 
        tr_row.head_id = tr_head.id
    GROUP BY
        verb,
        verb_compound,
        verb_tense,
        verb_mood,
        obj_case
    ORDER BY
        verb,
        verb_compound,
        freq DESC
    """
)
con.close()

In [14]:
con = sqlite3.connect(f"{RESULT_DATA_PATH}{TR_RESULT_STRICT}")

cur = con.cursor()

cur.execute("""
DROP TABLE IF EXISTS verbs_obj_cases_lemma_percentages
""")

cur.execute("""CREATE TABLE verbs_obj_cases_lemma_percentages AS
SELECT
    verb,
    verb_compound,
    verb_tense,
    verb_mood,

    SUM(freq) AS total,

    100.0 * SUM(CASE WHEN obj_case = 'nom'
                     THEN freq ELSE 0 END)
         / SUM(freq) AS percent_nom,

    100.0 * SUM(CASE WHEN obj_case = 'gen'
                     THEN freq ELSE 0 END)
         / SUM(freq) AS percent_gen,

    100.0 * SUM(CASE WHEN obj_case = 'part'
                     THEN freq ELSE 0 END)
         / SUM(freq) AS percent_par

FROM 
    verbs_obj_cases_lemma_freq
GROUP BY
    verb,
    verb_compound,
    verb_tense,
    verb_mood
"""
)

con.close()

### IV Verbs, obj cases examples

In [10]:
# all
con = sqlite3.connect(f"{RESULT_DATA_PATH}{TR_RESULT_STRICT}")

con.create_function("get_verb_tense", 1, get_verb_tense)
con.create_function("get_case", 1, get_case)
con.create_function("get_verb_mood", 1, get_verb_mood)
con.create_function("get_POS", 1, get_POS)

cur = con.cursor()
#cur.execute(f'ATTACH DATABASE "{SOURCE_DATA_PATH}{SOURCE_DATA}" AS source')
cur.execute(f'ATTACH DATABASE "{TR_SOURCE_DATA_PATH}{TR_DB}" AS tr')
cur.execute(f'ATTACH DATABASE "{TR_SOURCE_DATA_PATH}{SENTENCES_DB}" AS sents')

cur.execute("""
DROP TABLE IF EXISTS verb_strict_examples
""")

cur.execute("""
CREATE TABLE verb_strict_examples
AS
SELECT
    obj_cases.verb AS verb,
    obj_cases.verb_compound AS verb_compound,
    obj_cases.verb_tense AS verb_tense,
    obj_cases.verb_mood AS verb_mood,
    tr_row.form AS form,
    tr_row.lemma AS lemma,
    tr_row.loc AS loc,
    get_POS(tr_row.form) AS POS,
    get_case(tr_row.feats) AS current_case,
    sentences.id AS sentence_id,
    sentences.text AS sentence
FROM
    tr.transaction_row AS tr_row
INNER JOIN
    tr.transaction_head AS tr_head
    ON tr_row.head_id = tr_head.id
INNER JOIN
    verbs_obj_cases_lemma_percentages AS obj_cases
    ON tr_head.verb = obj_cases.verb
    AND tr_head.verb_compound = obj_cases.verb_compound
    AND get_verb_tense(tr_head.feats) = obj_cases.verb_tense
    AND get_verb_mood(tr_head.feats) = obj_cases.verb_mood
INNER JOIN
    sents.sentences AS sentences
    ON sentences.id = tr_head.sentence_id
WHERE
    tr_row.deprel = "obj"
GROUP BY
    obj_cases.verb,
    obj_cases.verb_compound,
    obj_cases.verb_tense,
    obj_cases.verb_mood,
    current_case
""")

con.close()

### V Examples for 10 random verbs

Getting examples for 10 random verbs, their lemmas and obj cases.

In [ ]:
EXAMPLE_RES = "verb_obj_case_examples_10.db"

con = sqlite3.connect(f"{RESULT_DATA_PATH}{EXAMPLE_RES}")

cur = con.cursor()
cur.execute(f'ATTACH DATABASE "{RESULT_DATA_PATH}{TR_RESULT_STRICT}" AS source')

cur.execute("""
DROP TABLE IF EXISTS random_10_verbs
""")

cur.execute("""
CREATE TABLE random_10_verbs AS
SELECT
    verb,
    verb_compound,
    verb_tense,
    verb_mood
FROM (
    SELECT
        verb,
        verb_compound,
        verb_tense,
        verb_mood,
        ROW_NUMBER() OVER (
            PARTITION BY
                verb,
                verb_compound,
                verb_tense,
                verb_mood
            ORDER BY RANDOM()
        ) AS rn
    FROM
        source.verb_strict_examples
)
WHERE
    rn = 1
ORDER BY
    RANDOM()
LIMIT 10
""")

con.close()

In [ ]:
con = sqlite3.connect(f"{RESULT_DATA_PATH}{EXAMPLE_RES}")

cur = con.cursor()
cur.execute(f'ATTACH DATABASE "{RESULT_DATA_PATH}{TR_RESULT_STRICT}" AS source')

cur.execute("""
DROP TABLE IF EXISTS random_10_verbs_with_stats
""")

cur.execute("""
CREATE TABLE random_10_verbs_with_stats AS
SELECT
    r.*,
    stats.total,
    stats.percent_nom,
    stats.percent_gen,
    stats.percent_par
FROM
    random_10_verbs AS r
LEFT JOIN
    source.verbs_obj_cases_lemma_percentages AS stats
    ON r.verb = stats.verb
   AND r.verb_compound = stats.verb_compound
   AND r.verb_tense = stats.verb_tense
   AND r.verb_mood = stats.verb_mood;
""")

con.close()

In [13]:
con = sqlite3.connect(f"{RESULT_DATA_PATH}{EXAMPLE_RES}")

cur = con.cursor()
cur.execute(f'ATTACH DATABASE "{RESULT_DATA_PATH}{TR_RESULT_STRICT}" AS source')

cur.execute("""
DROP TABLE IF EXISTS random_10_verbs_all_rows
""")

cur.execute("""
CREATE TABLE random_10_verbs_all_rows AS
SELECT
    npse.*,
    r.total,
    r.percent_nom,
    r.percent_gen,
    r.percent_par
FROM
    source.verb_strict_examples AS npse
INNER JOIN
    random_10_verbs_with_stats AS r
    ON npse.verb = r.verb
    AND npse.verb_compound = r.verb_compound
    AND npse.verb_tense = r.verb_tense
    AND npse.verb_mood = r.verb_mood

""")

con.close()